# Model to transform the gas to an hydrogen grid

Import packages

In [94]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

Import Data

In [95]:
# Specify the path to your Excel file
input_file_path = '../01_data/01_input_data/02_processed/'
excel_file_path = 'Data_update.xlsx'  

# Read the Excel file into a DataFrame
df_nodes = pd.read_excel(input_file_path + excel_file_path, sheet_name='Nodes')
df_commodities = pd.read_excel(input_file_path + excel_file_path, sheet_name='Commodities')
df_edges = pd.read_excel(input_file_path + excel_file_path, sheet_name='Edges')
df_parameter = pd.read_excel(input_file_path + excel_file_path, sheet_name='Parameters')
df_supply_values = pd.read_excel(input_file_path + excel_file_path, sheet_name='Supply')
df_demand_values = pd.read_excel(input_file_path + excel_file_path, sheet_name='Demand')

Create input data structure

In [108]:
# Extract nodes, edges and commodities from the DataFrames
All_nodes = df_nodes['Nodes'].dropna().tolist()
#Demand_nodes = df_nodes['Demand Nodes'].dropna().tolist()
Commodities = df_commodities['Commodities'].dropna().tolist()
Edges = list(zip(df_edges['Source'], df_edges['Destination']))

# Create a nested dictionary for initial capacities
Initial_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    initial_capacity = row['initial_capacities']

    edge = f"{source}{destination}"

    if commodity not in Initial_capacities:
        Initial_capacities[commodity] = {}

    Initial_capacities[commodity][edge] = initial_capacity

# Create a nested dictionary for max capacities
Max_capacities = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    max_capacity = row['max_capacities']

    edge = f"{source}{destination}"

    if commodity not in Max_capacities:
        Max_capacities[commodity] = {}

    Max_capacities[commodity][edge] = max_capacity

# Create a nested dictionary for edge cost
Edge_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    edge_cost = row['costs_edge']

    edge = f"{source}{destination}"

    if commodity not in Edge_cost:
        Edge_cost[commodity] = {}

    Edge_cost[commodity][edge] = edge_cost

# Create a nested dictionary for new pipelines
Pipe_new_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    new_cost = row['new_build_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_new_cost:
        Pipe_new_cost[commodity] = {}

    Pipe_new_cost[commodity][edge] = new_cost

# Create a nested dictionary for pipeline conversion
Pipe_conv_cost = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cost = row['conversion_cost']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_cost:
        Pipe_conv_cost[commodity] = {}

    Pipe_conv_cost[commodity][edge] = conv_cost

# Create a nested dictionary for adjusting the capacity when pipeline conversion
Pipe_conv_factor = {}
for index, row in df_parameter.iterrows():
    commodity = row['Commodity']
    source = row['Source']
    destination = row['Destination']
    conv_cap_factor = row['conversion_capacity_factor']

    edge = f"{source}{destination}"

    if commodity not in Pipe_conv_factor:
        Pipe_conv_factor[commodity] = {}

    Pipe_conv_factor[commodity][edge] = conv_cap_factor

# Create a nested dictionary for supply values, skipping 0 and NaN values
Supply_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    supply_node = row['Node']
    supply_value = row['Supply']

    if commodity not in Supply_values:
        Supply_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(supply_value) and supply_value != 0:
        Supply_values[commodity][supply_node] = supply_value

# Create a nested dictionary for demand values, skipping 0 and NaN values
Demand_values = {}
for index, row in df_demand_values.iterrows():
    commodity = row['Commodity']
    demand_node = row['Node']
    demand_value = row['Demand']

    if commodity not in Demand_values:
        Demand_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(demand_value) and demand_value != 0:
        Demand_values[commodity][demand_node] = demand_value

# Create a nested dictionary for node values, skipping 0 and NaN values
Node_values = {}
for index, row in df_supply_values.iterrows():
    commodity = row['Commodity']
    demand_node = row['Node']
    node_value = row['Supply']

    if commodity not in Node_values:
        Node_values[commodity] = {}

    # Skip 0 and NaN values
    if not pd.isna(node_value) and node_value != 0:
        Node_values[commodity][demand_node] = node_value

Print data structure for control

In [111]:
# Print the data
print("All Nodes:", All_nodes)
print("Commodities:", Commodities)
print("Edges:", Edges)
print("Initial Capacities:", Initial_capacities)
print("Max Capacities:", Max_capacities)
print("Costs per edge Capacities:", Edge_cost)
print("Costs for new pipelines:", Pipe_new_cost)
print("Costs for convert pipelines:", Pipe_conv_cost)
print("Conversion capacity factor:", Pipe_conv_factor)
print("Node Values:", Node_values)

All Nodes: ['S1', 'S2', 'S3', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10']
Commodities: ['Methane', 'Hydrogen']
Edges: [('S1', 'D1'), ('S1', 'D2'), ('S2', 'D2'), ('S2', 'D3'), ('S3', 'D1'), ('D1', 'D2'), ('D3', 'D4'), ('D4', 'D5'), ('D4', 'D6'), ('D6', 'D7'), ('D5', 'D8'), ('D5', 'D9'), ('D9', 'D10'), ('D7', 'D10')]
Initial Capacities: {'Methane': {'S1D1': 500, 'S1D2': 500, 'S2D2': 500, 'S2D3': 500, 'S3D1': 500, 'D1D2': 500, 'D3D4': 500, 'D4D5': 500, 'D4D6': 500, 'D6D7': 500, 'D5D8': 500, 'D5D9': 500, 'D9D10': 500, 'D7D10': 500}, 'Hydrogen': {'S1D1': 500, 'S1D2': 500, 'S2D2': 500, 'S2D3': 500, 'S3D1': 500, 'D1D2': 500, 'D3D4': 500, 'D4D5': 500, 'D4D6': 500, 'D6D7': 500, 'D5D8': 500, 'D5D9': 500, 'D9D10': 500, 'D7D10': 500}}
Max Capacities: {'Methane': {'S1D1': 500, 'S1D2': 500, 'S2D2': 500, 'S2D3': 500, 'S3D1': 500, 'D1D2': 500, 'D3D4': 500, 'D4D5': 500, 'D4D6': 500, 'D6D7': 500, 'D5D8': 500, 'D5D9': 500, 'D9D10': 500, 'D7D10': 500}, 'Hydrogen': {'S1D1': 500, 'S1D2': 50

In [98]:
#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = Pipe_conv_factor

Create model

In [100]:
# Create a new model
model = gp.Model("Grid_Transformation")

Define parameters

In [112]:
# Parameters
supply_nodes = Supply_nodes  # Supply nodes
demand_nodes = Demand_nodes  # Demand nodes
all_nodes = All_nodes # Nodes of the system
commodities = Commodities  # Commodity types
edges = Edges  # Edges
initial_capacities = Initial_capacities # Initial capacities
max_capacities = Max_capacities  # Maximum capacities
costs_edge = Edge_cost  # Cost to transport from node to node
capacity_new_cost = Pipe_new_cost  # Cost to increase capacity
capacity_change_cost = Pipe_conv_cost  # Cost to increase capacity

supply_values = Supply_values  # Supply values
demand_values = Demand_values  # Demand values
node_value = Supply_values

Define decision variables

In [113]:
# Decision variables
x_flow = {} #flow of commodity on an edge
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"w_{commodity}_{edge[0]}_{edge[1]}")  

Define objective and constraints

In [114]:
# Objective function (minimize total transportation cost + cost to increase capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges) +
    gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges) +
    gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in edges),
    GRB.MINIMIZE
)

# Constraints

#Option to fix the problem. missing inflow equals outflow constraint
for node in all_nodes:  # change the supply_nodes and demand_nodes into a set of nodes with positive (supply) or negative (demand) values
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in edges if edge[1] == node) 
                                    - gp.quicksum(x_flow[commodity][edge] for edge in edges if edge[0] == node)
                                    + node_value[commodity][node] 
                                    == 0, f"flow_constraint_{commodity}_{node}")

#Capacity constraint for flow
for commodity in commodities:
    for edge in edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")

# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"total_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraint for edge conversion
for edge in edges:
    model.addConstr(Change[Commodities[0]][edge] + Change[Commodities[1]][edge] == 1, "switching_constraint")
for commodity in commodities:
    for edge in edges:
        model.addConstr(initial_capacities[Commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity][edge] #* conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

Optimize the model

In [115]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11.0 (22621.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 324 rows, 224 columns and 662 nonzeros
Model fingerprint: 0x459110b6
Variable types: 168 continuous, 56 integer (56 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+02]
  Objective range  [5e-01, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]

MIP start from previous solve produced solution with objective 5467 (0.03s)
Loaded MIP start from previous solve with objective 5467

Presolve removed 324 rows and 224 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.04 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 5467 

Optimal solution found (tolerance 1.00e-04)
Best objective 5.467

Results processing

In [116]:
# Print the results
if model.status == GRB.OPTIMAL:
    print("Optimal solution found!")
    for commodity in commodities:
        for edge in edges:
            print(f"{commodity}, {edge}: Flow of Commodity = {x_flow[commodity][edge].x}, New Capacity = {y_new_cap[commodity][edge].x}, Switched = {Change[commodity][edge].x}, Changed Capacity = {z_conv_cap[commodity][edge].x}")
    print("****************************")
    print(f"Total cost: {model.objVal}")
else:
    print("No optimal solution found.")

Optimal solution found!
Methane, ('S1', 'D1'): Flow of Commodity = 0.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0
Methane, ('S1', 'D2'): Flow of Commodity = 20.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0
Methane, ('S2', 'D2'): Flow of Commodity = 0.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0
Methane, ('S2', 'D3'): Flow of Commodity = 81.0, New Capacity = 81.0, Switched = 0.0, Changed Capacity = 0.0
Methane, ('S3', 'D1'): Flow of Commodity = 15.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0
Methane, ('D1', 'D2'): Flow of Commodity = 0.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity = 500.0
Methane, ('D3', 'D4'): Flow of Commodity = 56.0, New Capacity = 56.0, Switched = 0.0, Changed Capacity = 0.0
Methane, ('D4', 'D5'): Flow of Commodity = 40.0, New Capacity = 40.0, Switched = 0.0, Changed Capacity = 0.0
Methane, ('D4', 'D6'): Flow of Commodity = 15.0, New Capacity = 0.0, Switched = 1.0, Changed Capacity 

In [106]:
# Assuming commodities, edges, x_flow, y_new_cap, Change, and z_conv_cap are defined in your code

# Create lists to store the data
results_data = []
columns = ["Commodity", "Edge", "Flow", "New Capacity", "Switched", "Changed Capacity"]

# Check if the model has an optimal solution
if model.status == GRB.OPTIMAL:
    for commodity in commodities:
        for edge in edges:
            # Append data to the list
            results_data.append([commodity, edge, x_flow[commodity][edge].x, y_new_cap[commodity][edge].x, Change[commodity][edge].x, z_conv_cap[commodity][edge].x])
else:
    print("No optimal solution found.")
    
# Create a DataFrame
results_df = pd.DataFrame(results_data, columns=columns)

# Print the DataFrame
print(results_df)

   Commodity       Edge  Flow  New Capacity  Switched  Changed Capacity
0    Methane   (S1, D1)   0.0           0.0       1.0             500.0
1    Methane   (S1, D2)  20.0           0.0       1.0             500.0
2    Methane   (S2, D2)   0.0           0.0       1.0             500.0
3    Methane   (S2, D3)  81.0          81.0       0.0               0.0
4    Methane   (S3, D1)  15.0           0.0       1.0             500.0
5    Methane   (D1, D2)   0.0           0.0       1.0             500.0
6    Methane   (D3, D4)  56.0          56.0       0.0               0.0
7    Methane   (D4, D5)  40.0          40.0       0.0               0.0
8    Methane   (D4, D6)  15.0           0.0       1.0             500.0
9    Methane   (D6, D7)  10.0           0.0       1.0             500.0
10   Methane   (D5, D8)  10.0           0.0       1.0             500.0
11   Methane   (D5, D9)  10.0           0.0       1.0             500.0
12   Methane  (D9, D10)   0.0           0.0       1.0           